In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

In [33]:
BASE_DIR = Path.cwd().parent # Assuming this notebook is in a 'notebooks' subdirectory
RAW_DIR = BASE_DIR / 'data' / 'raw' # Define raw data directory
FEATURED_DIR = BASE_DIR / 'data' / 'featured' # Define featured data directory
PLOTS_DIR = BASE_DIR / 'data' / 'plots' # Define plots directory
FEATURED_DIR.mkdir(parents=True, exist_ok=True) # Create featured data directory if it doesn't exist
PLOTS_DIR.mkdir(parents=True, exist_ok=True) # Create plots directory if it doesn't exist

In [34]:
ASSETS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META','TSLA', 'NVDA','AMD','IBM','BTC-USD','ETH-USD'] # List of assets to analyze

In [ ]:
def load_raw(ticker): 
    """Load raw data for a given ticker."""
    safe = ticker.replace('-', '_')  # Sanitize ticker for file naming
    file_path = RAW_DIR / f'{safe}.csv' # Construct file path
    df   = pd.read_csv(file_path,index_col='Date', parse_dates=True)	# Load CSV with Date as index
    df.index.name = 'Date' # Ensure index is named 'Date'
    df.columns = [c.strip() for c in df.columns] # Strip whitespace from column names
    return df

In [39]:
assets = {t: load_raw(t) for t in ASSETS} # Load raw data for all assets into a dictionary

ValueError: 'Date' is not in list

In [ ]:
# cleaning
for ticker, df in assets.items():                   # Loop through each asset's DataFrame
    print(f'\n=== {ticker} ===')                    # Basic EDA outputs
    print(f'Shape       : {df.shape}')              # Rows & columns
    print(f'Date range  : {df.index.min()} to {df.index.max()}') # Date range
    print(f'Missing vals: {df.isnull().sum().sum()}')           # Total missing values
    print(f'Duplicates  : {df.duplicated().sum()}')            #    Duplicate rows
    print(f'Price range : ${df["Close"].min():.2f} - ${df["Close"].max():.2f}') # Price range
    # Flag extreme single-day returns (potential data errors)       
    df['tmp_ret'] = df['Close'].pct_change()            # Calculate daily returns in a temporary column
    anomalies = df[df['tmp_ret'].abs() > 0.50]
    if not anomalies.empty:
        print(f'ANOMALIES   : {len(anomalies)} rows > 50% daily move')
    df.drop(columns=['tmp_ret'], inplace=True)    # Clean up temporary column


In [ ]:
# Handle missing values
def handle_missing(df,ticker):
    """Handle missing values in the DataFrame."""
    df = df.fill()  # Placeholder for actual imputation strategy (e.g., forward fill, interpolation)
    df = df.dropna()  # Drop any remaining missing values
    df = df[~df.index.duplicated(keep='first')]  # Remove duplicate rows based on index
    df = df.sort_index()  # Ensure data is sorted by date
    print(f'{ticker}: Missing values handled, duplicates removed, data sorted.')
    return df

In [ ]:
# Descriptive statistics
for ticker, df in assets.items():   # Loop through each asset's DataFrame
    df['Daily_Return'] = df['Close'].pct_change()
    print(f'\n--- {ticker} ---')
    print(df['Close'].describe().round(2))
    print(f'Avg daily return : {df["Daily_Return"].mean():.4f}')
    print(f'Std daily return : {df["Daily_Return"].std():.4f}')
    print(f'Best day         : {df["Daily_Return"].max():.2%}')
    print(f'Worst day        : {df["Daily_Return"].min():.2%}')
    annualised_vol = df['Daily_Return'].std() * (252 ** 0.5)
    print(f'Annualised vol   : {annualised_vol:.2%}')


In [ ]:
# Time series analysis
# Normalize prize history
fig, axes = plt.subplots(len(ASSETS), 1, figsize=(12, 3*len(ASSETS)), sharex=True) 
for i, ticker in enumerate(ASSETS):
    df = assets[ticker]
    df['Norm_Close'] = df['Close'] / df['Close'].iloc[0]  # Normalize to 1 at start
    axes[i].plot(df.index, df['Norm_Close'], label=ticker)
    axes[i].set_title(f'{ticker} Normalized Price History')
    axes[i].set_ylabel('Normalized Price')
    axes[i].legend()
plt.xlabel('Date')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'normalized_price_history.png')
plt.show()

In [ ]:
# Volume over time
for ticker, df in assets.items():
    plt.figure(figsize=(12, 4))
    plt.plot(df['Volume'], label='Volume', color='blue', linewidth=1)
    plt.title(f'{ticker} Trading Volume Over Time')
    plt.xlabel('Date')
    plt.ylabel('Volume')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'data/plots/{ticker}_volume.png', dpi=150)
    plt.show()

In [ ]:
# Daily returns distribution
fig, axes = plt.subplots(len(ASSETS), 1, figsize=(12, 3*len(ASSETS)), sharex=True)
for (ticker, df), ax in zip(assets.items(), axes):
    df['Daily_Return'] = df['Close'].pct_change()
    sns.histplot(df['Daily_Return'].dropna(), bins=50, kde=True, ax=ax, color='salmon')
    ax.set_title(f'{ticker} Daily Returns Distribution', fontsize=14)
    ax.set_xlabel('Daily Return')
    ax.set_ylabel('Frequency')
    ax.grid(alpha=0.3)
plt.suptitle('Daily Returns Distribution', fontsize=16)
plt.tight_layout()
plt.savefig('data/plots/02_daily_returns.png', dpi=150)
plt.show()

In [ ]:
# 30-Day Rolling Annualised Volatility
fig, axes = plt.subplots(len(ASSETS), 1, figsize=(12, 3*len(ASSETS)), sharex=True)
for (ticker, df), ax in zip(assets.items(), axes):
    df['Daily_Return'] = df['Close'].pct_change()
    df['Rolling_Std'] = df['Daily_Return'].rolling(window=30).std()
    df['Rolling_Ann_Vol'] = df['Rolling_Std'] * (252 ** 0.5)
    ax.plot(df.index, df['Rolling_Ann_Vol'], color='purple', linewidth=1.5)
    ax.set_title(f'{ticker} 30-Day Rolling Annualised Volatility', fontsize=14)
    ax.set_ylabel('Annualised Volatility')
    ax.grid(alpha=0.3)
plt.suptitle('30-Day Rolling Annualised Volatility', fontsize=16)
plt.tight_layout()
plt.savefig('data/plots/03_rolling_volatility.png', dpi=150)
plt.show()

In [ ]:
# Correlation analysis
df2 = pd.DataFrame({ticker: df['Close'] for ticker, df in assets.items()})
corr_matrix = df2.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Closing Prices', fontsize=16)
plt.tight_layout()
plt.savefig('data/plots/04_correlation_matrix.png', dpi=150)
plt.show()

# finding correlations
corr_matrix = df2.corr()
print("Correlation Matrix:")
print(corr_matrix)


In [ ]:
# K-Means clustering on returns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# feature matrix : one row per asset, columns are features like mean return, volatility, etc.
features = []
for ticker, df in assets.items():
    df['Daily_Return'] = df['Close'].pct_change()
    features.append({
        'Asset': ticker,
        'Mean_Return': df['Daily_Return'].mean(),
        'Volatility': df['Daily_Return'].std(),
        'Max_Return': df['Daily_Return'].max(),
        'Min_Return': df['Daily_Return'].min(),
        "Avg_Volume": df['Volume'].mean(),
        "Avg_Price": df['Close'].mean()
    })
    
    
features_df = pd.DataFrame(features).set_index('Asset')
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_df)
# K-means clustering
kmeans = KMeans(n_clusters=3, random_state=42)
features_df['Cluster'] = kmeans.fit_predict(scaled_features)

print("\nAsset Clusters:")
print(features_df[['Cluster', 'Mean_Return', 'Volatility', 'Avg_Volume', 'Avg_Price']])


# Visualize clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(data=features_df, x='Mean_Return', y='Volatility', hue='Cluster', palette='Set2', s=100)
for i in range(features_df.shape[0]):
    plt.text(features_df['Mean_Return'][i], features_df['Volatility'][i], features_df.index[i], fontsize=9)
plt.title('Asset Clusters Based on Mean Return and Volatility', fontsize=16)
plt.xlabel('Mean Daily Return')
plt.ylabel('Volatility')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('data/plots/05_asset_clusters.png', dpi=150)
plt.show()

In [ ]:
# feature engineering
import ta

def add_technical_indicators(df):
    """Add technical indicators to the DataFrame."""
    # moving averages
    df['SMA_20'] = ta.trend.sma_indicator(df['Close'], window=20)
    df['SMA_50'] = ta.trend.sma_indicator(df['Close'], window=50)
    df['EMA_12'] = ta.trend.ema_indicator(df['Close'], window=12)
    df['EMA_26'] = ta.trend.ema_indicator(df['Close'], window=26)
    # momentum indicators
    # RSI
    df['RSI_14'] = ta.momentum.rsi(df['Close'], window=14)
    # MACD
    df['MACD'] = ta.trend.macd(df['Close'], window_slow=26, window_fast=12, window_signal=9)
    df['MACD_Signal'] = ta.trend.macd_signal(df['Close'])
    df['MACD_Hist'] = ta.trend.macd_diff(df['Close'])
    
    # Bollinger Bands
    bb = ta.volatility.BollingerBands(df['Close'], window=20, window_dev=2)
    df["BB_High"] = bb.bollinger_hband()
    df["BB_Low"] = bb.bollinger_lband()
    df["BB_Mid"] = bb.bollinger_mavg()
    df["BB_Width"] = bb.bollinger_wband()
    
    # Daily returns & volatility
    df['Daily_Return'] = df['Close'].pct_change()
    df['Volatility_20'] = df['Daily_Return'].rolling(window=20).std()
    
    # Derived signal columns
    df['SMA20_Signal'] = (df['Close'] > df['SMA_20']).astype(int)
    df['RSI_Signal'] = (df['RSI_14'] < 30).astype(int) - (df['RSI_14'] > 70).astype(int) # 1=oversold, -1=overbought
    df['MACD_Signal'] = (df['MACD'] > df['MACD_Signal']).astype(int)
    df['MA_Cross'] = (df['SMA_20'] > df['SMA_50']).astype(int)
    
    
    df.dropna(inplace=True) # Drop rows with NaN values created by indicators

    return df


# Apply feature engineering to all assets
featured_assets = {}
for ticker, df in assets.items():
    featured_df = add_technical_indicators(df)
    featured_assets[ticker] = featured_df
    # Save featured data to CSV
    safe = ticker.replace('-', '_')
    featured_df.to_csv(FEATURED_DIR / f'{safe}_featured.csv')
    print(f'Processed {ticker}: {featured_df.shape[0]} rows, {featured_df.shape[1]} columns')

In [ ]:
# indicator visualization
# run for each asset
df = featured_assets['AAPL'] 
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
# Price, MA20, MA50 , Bollinger Bands
axes[0].plot(df.index, df['Close'], label='Close Price', color='blue')
axes[0].plot(df.index, df['SMA_20'], label='SMA 20', color='orange')
axes[0].plot(df.index, df['SMA_50'], label='SMA 50', color='green')
axes[0].plot(df.index, df['BB_High'], label='BB High', color='red', linestyle='--')
axes[0].plot(df.index, df['BB_Low'], label='BB Low', color='red', linestyle='--')
axes[0].set_title('Price with SMA and Bollinger Bands')     
axes[0].set_ylabel('Price')
axes[0].legend()            

# RSI
axes[1].plot(df.index, df['RSI_14'], label='RSI 14', color='purple')
axes[1].axhline(70, color='red', linestyle='--')    
axes[1].axhline(30, color='green', linestyle='--')    
axes[1].set_title('RSI')
axes[1].set_ylabel('RSI')
axes[1].legend()            

# Volume
axes[2].bar(df.index, df['Volume'], label='Volume', color='gray')
axes[2].set_title('Trading Volume')
axes[2].set_ylabel('Volume')
axes[2].legend()            


#MACD
axes[3].plot(df.index, df['MACD'], label='MACD', color='brown')
axes[3].plot(df.index, df['MACD_Signal'], label='Signal Line', color='cyan')
axes[3].bar(df.index, df['MACD_Hist'], label='MACD Histogram', color='magenta')
axes[3].set_title('MACD')   
axes[3].set_ylabel('MACD')
axes[3].legend()

plt.suptitle('AAPL Technical Indicators', fontsize=16)
plt.xlabel('Date')
plt.tight_layout()
plt.savefig('data/plots/AAPL_technical_indicators.png', dpi=150)
plt.show()


In [ ]:
# indicator visualization
# run for each asset
df = featured_assets['MSFT'] 
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
# Price, MA20, MA50 , Bollinger Bands
axes[0].plot(df.index, df['Close'], label='Close Price', color='blue')
axes[0].plot(df.index, df['SMA_20'], label='SMA 20', color='orange')
axes[0].plot(df.index, df['SMA_50'], label='SMA 50', color='green')
axes[0].plot(df.index, df['BB_High'], label='BB High', color='red', linestyle='--')
axes[0].plot(df.index, df['BB_Low'], label='BB Low', color='red', linestyle='--')
axes[0].set_title('Price with SMA and Bollinger Bands')     
axes[0].set_ylabel('Price')
axes[0].legend()            

# RSI
axes[1].plot(df.index, df['RSI_14'], label='RSI 14', color='purple')
axes[1].axhline(70, color='red', linestyle='--')    
axes[1].axhline(30, color='green', linestyle='--')    
axes[1].set_title('RSI')
axes[1].set_ylabel('RSI')
axes[1].legend()            

# Volume
axes[2].bar(df.index, df['Volume'], label='Volume', color='gray')
axes[2].set_title('Trading Volume')
axes[2].set_ylabel('Volume')
axes[2].legend()            


#MACD
axes[3].plot(df.index, df['MACD'], label='MACD', color='brown')
axes[3].plot(df.index, df['MACD_Signal'], label='Signal Line', color='cyan')
axes[3].bar(df.index, df['MACD_Hist'], label='MACD Histogram', color='magenta')
axes[3].set_title('MACD')   
axes[3].set_ylabel('MACD')
axes[3].legend()

plt.suptitle('MSFT Technical Indicators', fontsize=16)
plt.xlabel('Date')
plt.tight_layout()
plt.savefig('data/plots/MSFT_technical_indicators.png', dpi=150)
plt.show()
